In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np

In [4]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [5]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [6]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


In [7]:
# build a char matrix to find words without given letters
char_matrix = np.zeros(shape = (word_df.shape[0], 26), dtype = np.int8)
letter_dict = {l:p for p,l in enumerate(ascii_lowercase)}
for i_row, row in word_df.iterrows():
    for l in row['lcase']:
        char_matrix[i_row, letter_dict[l]] += 1


In [8]:
# sum by column...
# these are the words we can remove in order to find groups...
word_count_by_letter = char_matrix.sum(axis = 0)
word_count_by_letter

array([2545,  799, 1154, 1115, 2360,  575,  838, 1058, 1993,  226,  831,
       1508,  968, 1517, 1976,  874,   91, 1732, 1993, 1418, 1636,  405,
        603,  224, 1209,  237])

## EXAMPLES OF BYTE COMPARISONS

In [9]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [10]:
# no letters in common
w1b & w2b

0

In [11]:
# letters in common
w1b & w3b

147456

In [12]:
# bitwise or
w1b | w2b

673975

In [13]:
# this is the same as directly above
byte_encode_words('abhorcleft')

673975

In [14]:
byte_encode_words(ascii_lowercase)

67108863

# BUILD LEVEL 2

In [17]:
# find pairs, but exclude groups of letters
l2_dict = {} 
total_pairs = 0
for l,p in letter_dict.items():
    print(l)

    l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)

    curr_word_byte_list = word_byte_array[(char_matrix[:, p] == 0)]
    print(curr_word_byte_list.shape)
    row_index = 0
    
    for w1_be, w2_be in combinations(curr_word_byte_list, 2):
        if w1_be & w2_be == 0:   
            # they share no letters in common
            l2 = w1_be | w2_be                          
            
            l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)    
            row_index += 1

    # trim the data frame
    l2_list = l2_list[:row_index, :]
    print(l2_list.shape)
    l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])
    total_pairs += l2_df.shape[0]
    l2_dict[l] = l2_df

a
(3432,)
(871979, 3)
b
(5178,)
(2278449, 3)
c
(4823,)
(1899697, 3)
d
(4862,)
(1981467, 3)
e
(3617,)
(960869, 3)
f
(5402,)
(2510281, 3)
g
(5139,)
(2236432, 3)
h
(4919,)
(1970356, 3)
i
(3984,)
(1142477, 3)
j
(5751,)
(2926607, 3)
k
(5146,)
(2228669, 3)
l
(4469,)
(1609497, 3)
m
(5009,)
(2102934, 3)
n
(4460,)
(1606470, 3)
o
(4001,)
(1152892, 3)
p
(5103,)
(2193957, 3)
q
(5886,)
(3108792, 3)
r
(4245,)
(1449143, 3)
s
(3984,)
(1267714, 3)
t
(4559,)
(1705155, 3)
u
(4341,)
(1362578, 3)
v
(5572,)
(2755187, 3)
w
(5374,)
(2466174, 3)
x
(5753,)
(2937566, 3)
y
(4768,)
(1764530, 3)
z
(5740,)
(2929264, 3)


In [16]:
l2_df['l2'].unique().shape

(527636,)

# BUILD LEVELS 3 THROUGH 5

In [ ]:
# so, now, let's try computing all possible pairs
total_output = np.full(shape = (1000000, 9), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3 = (word_byte_array & l2) == 0

    # l3 words with different letters
    output_array_w3b = word_byte_array[positional_idx_l3]    

    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    ## enumerate level 3
    for w3b, l3 in zip(output_array_w3b, output_array_l3):

        # build level 4

        # l4 idx
        positional_idx_l4 = (word_byte_array & l3) == 0

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx_l4]    
        
        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        ## enumerate level 5
        for w4b, l4 in zip(output_array_w4b, output_array_l4):

            # build level 5

            # l5 idx
            positional_idx_l5 = (word_byte_array & l4) == 0
            
            # words with different letters
            output_array_w5b = word_byte_array[positional_idx_l5]    

            if output_array_w5b.size > 0:
                    
                # accumulated letters
                output_array_l5 = output_array_w5b | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b, output_array_l5):

                    temp_list = np.array([w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5], dtype = np.int32)
                    total_output[row_index, :] = temp_list                   
                    row_index += 1


    if i_row % 10000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)
    


# CREATE AND SAVE OUTPUT

In [ ]:
total_output = total_output[:row_index, :]
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5']
l5_df = pd.DataFrame(data = total_output, columns = col_names)


In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.tail()

In [ ]:
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)
